# 04 ? ??nh gi? m? h?nh ?? train

Notebook n?y n?p artifact do `03_train.ipynb` t?o ra (`model.joblib`, `decision_config.json`) v? ??nh gi? tr?n c?ng holdout stratified 20% (`random_state=42`). Threshold ???c ch?n t? 5-fold out-of-fold d? ?o?n c?a 80% train. Test ch? d?ng ?? b?o c?o; kh?ng ch?n threshold hay model t? test.

In [ ]:
import io
import json
import time
import zipfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
try:
    from google.colab import files
except ModuleNotFoundError:
    files = None

def read_csv_from_zip(zip_bytes):
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
        csv_name = next(name for name in archive.namelist() if name.lower().endswith('.csv'))
        with archive.open(csv_name) as csv_file:
            return pd.read_csv(csv_file)

if files is not None:
    print('Upload dataset (.csv/.zip), model.joblib v? decision_config.json')
    uploaded = files.upload()
    def uploaded_file(suffix):
        return next((name for name in uploaded if name.lower().endswith(suffix)), None)
    data_name, model_name, config_name = uploaded_file('.csv') or uploaded_file('.zip'), uploaded_file('.joblib'), uploaded_file('.json')
    if not all([data_name, model_name, config_name]):
        raise ValueError('C?n ?? dataset CSV/ZIP, model.joblib v? decision_config.json.')
    df = read_csv_from_zip(uploaded[data_name]) if data_name.lower().endswith('.zip') else pd.read_csv(io.BytesIO(uploaded[data_name]))
    models = joblib.load(io.BytesIO(uploaded[model_name]))
    decision_config = json.loads(uploaded[config_name].decode('utf-8'))
else:
    roots = [Path.cwd(), *Path.cwd().parents]
    ai_models_dir = next((p for root in roots for p in (root, root / 'ai-models') if p.name == 'ai-models' and p.is_dir()), None)
    if ai_models_dir is None:
        raise FileNotFoundError('Kh?ng t?m th?y ai-models/.')
    df = pd.read_csv(ai_models_dir / 'data' / 'diabetes.csv')
    models = joblib.load(ai_models_dir / 'models' / 'model.joblib')
    decision_config = json.loads((ai_models_dir / 'models' / 'decision_config.json').read_text(encoding='utf-8'))

feature_columns = [c for c in df.columns if c not in ('Outcome', 'index')]
X, y = df[feature_columns], df['Outcome'].astype(int)
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Test set: {len(y_test)} m?u | khuy?n ngh? CV: {decision_config["model_name"]}')

In [ ]:
def calculate_metrics(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, predictions, average='weighted', zero_division=0)
    return {
        'Accuracy': accuracy_score(y_true, predictions),
        'Weighted precision': precision,
        'Weighted recall': recall,
        'Weighted F1': f1,
        'Positive recall (sensitivity)': recall_score(y_true, predictions, pos_label=1, zero_division=0),
        'Specificity': recall_score(y_true, predictions, pos_label=0, zero_division=0),
        'ROC-AUC': roc_auc_score(y_true, probabilities),
    }

rows = []
for key, model in models.items():
    threshold = decision_config['models'][key]['threshold']
    start = time.perf_counter()
    probabilities = model.predict_proba(X_test)[:, 1]
    prediction_time_ms = (time.perf_counter() - start) * 1000
    test_metrics = calculate_metrics(y_test, probabilities, threshold)
    cv = decision_config['models'][key]['cv_metrics']
    rows.append({
        'Model': key, 'Threshold (CV)': threshold,
        'CV sensitivity': cv['positive_recall'], 'CV specificity': cv['specificity'],
        **{f'Test {name}': value for name, value in test_metrics.items()},
        'Prediction time (ms)': prediction_time_ms,
    })

results = pd.DataFrame(rows).sort_values(['CV specificity', 'CV sensitivity'], ascending=False)
display(results.style.format(precision=4))
print('Model ???c ?? xu?t theo CV:', decision_config['model_name'])

**L?u ?:** th?i gian d? ?o?n ph? thu?c m?i tr??ng ch?y. Sensitivity target 80% trong d? ?n ch? l? gi? ??nh minh h?a, kh?ng ph?i ng??ng l?m s?ng ?? x?c nh?n.